# Mamba SSM for Echocardiography Dehazing

## Overview
This notebook implements a **Mamba State Space Model (SSM)** based architecture for dehazing transthoracic echocardiographic images. The approach is inspired by the MICCAI Dehazing Echocardiography Challenge (DehazingEcho2025).

### Key Features:
- **Mamba SSM Architecture**: Leverages selective state space models for efficient long-range dependency modeling
- **U-Net-like Encoder-Decoder**: Combines Mamba blocks with skip connections for image restoration
- **Comprehensive Evaluation**: Includes FID, PSNR, SSIM, and perceptual metrics

### Dataset:
- **Input**: Noisy/Hazy echocardiography frames (`Dataset/noisy`)
- **Ground Truth**: Clean echocardiography frames (`Dataset/Output`)

### Model Architecture:
The model uses Vision Mamba (Vim) blocks in a U-Net structure for effective dehazing.

## 1. Install Dependencies and Import Libraries

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install mamba-ssm  # For CUDA-enabled Mamba (requires CUDA)
# !pip install einops timm scipy scikit-image pandas matplotlib tqdm
# !pip install pytorch-fid lpips

import os
import glob
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid

# For metrics
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from scipy import stats

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Configuration and Hyperparameters

In [ ]:
# Configuration
class Config:
    # Paths
    BASE_DIR = os.path.dirname(os.getcwd())  # Parent directory
    NOISY_DIR = os.path.join(BASE_DIR, 'Dataset', 'noisy')
    CLEAN_DIR = os.path.join(BASE_DIR, 'Dataset', 'Output')
    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints', 'mamba_dehazing')
    RESULTS_DIR = os.path.join(BASE_DIR, 'results', 'mamba_dehazing')
    
    # Model parameters
    IMG_SIZE = 256  # Image size (height, width)
    IN_CHANNELS = 1  # Grayscale echocardiography images
    BASE_DIM = 64  # Base number of channels
    MAMBA_DIM = 256  # Dimension for Mamba blocks
    SSM_STATE_DIM = 16  # State dimension for SSM
    SSM_EXPAND = 2  # Expansion factor
    NUM_MAMBA_LAYERS = 4  # Number of Mamba layers per block
    
    # Training parameters
    BATCH_SIZE = 8
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    SCHEDULER_PATIENCE = 10
    EARLY_STOP_PATIENCE = 20
    
    # Loss weights
    L1_WEIGHT = 1.0
    PERCEPTUAL_WEIGHT = 0.1
    SSIM_WEIGHT = 0.5
    
    # Data split
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.1
    TEST_RATIO = 0.1
    
    # Logging
    LOG_INTERVAL = 10
    SAVE_INTERVAL = 5

config = Config()

# Create directories
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)

print("Configuration loaded successfully!")
print(f"Noisy images directory: {config.NOISY_DIR}")
print(f"Clean images directory: {config.CLEAN_DIR}")
print(f"Checkpoint directory: {config.CHECKPOINT_DIR}")

## 3. Dataset and Data Loading

In [ ]:
class EchoDehazeDataset(Dataset):
    """Dataset for echocardiography dehazing with paired noisy-clean images."""
    
    def __init__(self, noisy_dir, clean_dir, transform=None, file_list=None):
        """
        Args:
            noisy_dir: Directory containing noisy/hazy images
            clean_dir: Directory containing clean/ground truth images
            transform: Transforms to apply
            file_list: List of filenames to use (for train/val/test split)
        """
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.transform = transform
        
        # Get all image files
        if file_list is not None:
            self.files = file_list
        else:
            noisy_files = set(os.listdir(noisy_dir))
            clean_files = set(os.listdir(clean_dir))
            # Only use files that exist in both directories
            self.files = sorted(list(noisy_files.intersection(clean_files)))
        
        print(f"Dataset initialized with {len(self.files)} paired images")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        filename = self.files[idx]
        
        # Load images
        noisy_path = os.path.join(self.noisy_dir, filename)
        clean_path = os.path.join(self.clean_dir, filename)
        
        noisy_img = Image.open(noisy_path).convert('L')  # Convert to grayscale
        clean_img = Image.open(clean_path).convert('L')
        
        # Apply transforms
        if self.transform:
            noisy_img = self.transform(noisy_img)
            clean_img = self.transform(clean_img)
        
        return {
            'noisy': noisy_img,
            'clean': clean_img,
            'filename': filename
        }


def get_data_transforms(img_size, is_train=True):
    """Get data transforms for training and evaluation."""
    if is_train:
        transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize to [-1, 1]
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
    return transform


def create_dataloaders(config):
    """Create train, validation, and test dataloaders."""
    # Get all paired files
    noisy_files = set(os.listdir(config.NOISY_DIR))
    clean_files = set(os.listdir(config.CLEAN_DIR))
    all_files = sorted(list(noisy_files.intersection(clean_files)))
    
    print(f"Total paired images found: {len(all_files)}")
    
    # Group files by patient for proper splitting (avoid data leakage)
    patient_files = {}
    for f in all_files:
        # Extract patient ID from filename (e.g., "patient-1-4C-frame-1.png")
        parts = f.split('-')
        if len(parts) >= 2:
            patient_id = f"{parts[0]}-{parts[1]}"
            if patient_id not in patient_files:
                patient_files[patient_id] = []
            patient_files[patient_id].append(f)
    
    patients = list(patient_files.keys())
    random.shuffle(patients)
    
    # Split patients
    n_train = int(len(patients) * config.TRAIN_RATIO)
    n_val = int(len(patients) * config.VAL_RATIO)
    
    train_patients = patients[:n_train]
    val_patients = patients[n_train:n_train + n_val]
    test_patients = patients[n_train + n_val:]
    
    # Get file lists for each split
    train_files = [f for p in train_patients for f in patient_files[p]]
    val_files = [f for p in val_patients for f in patient_files[p]]
    test_files = [f for p in test_patients for f in patient_files[p]]
    
    print(f"Train: {len(train_files)} images from {len(train_patients)} patients")
    print(f"Val: {len(val_files)} images from {len(val_patients)} patients")
    print(f"Test: {len(test_files)} images from {len(test_patients)} patients")
    
    # Create datasets
    train_transform = get_data_transforms(config.IMG_SIZE, is_train=True)
    eval_transform = get_data_transforms(config.IMG_SIZE, is_train=False)
    
    train_dataset = EchoDehazeDataset(
        config.NOISY_DIR, config.CLEAN_DIR, 
        transform=train_transform, file_list=train_files
    )
    val_dataset = EchoDehazeDataset(
        config.NOISY_DIR, config.CLEAN_DIR,
        transform=eval_transform, file_list=val_files
    )
    test_dataset = EchoDehazeDataset(
        config.NOISY_DIR, config.CLEAN_DIR,
        transform=eval_transform, file_list=test_files
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, batch_size=config.BATCH_SIZE, 
        shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=config.BATCH_SIZE,
        shuffle=False, num_workers=0, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config.BATCH_SIZE,
        shuffle=False, num_workers=0, pin_memory=True
    )
    
    return train_loader, val_loader, test_loader, test_files


# Create dataloaders
train_loader, val_loader, test_loader, test_files = create_dataloaders(config)

## 4. Visualize Sample Data

In [ ]:
def visualize_samples(dataloader, num_samples=4):
    """Visualize noisy and clean image pairs."""
    batch = next(iter(dataloader))
    noisy = batch['noisy'][:num_samples]
    clean = batch['clean'][:num_samples]
    filenames = batch['filename'][:num_samples]
    
    fig, axes = plt.subplots(2, num_samples, figsize=(4*num_samples, 8))
    
    for i in range(num_samples):
        # Denormalize images for visualization
        noisy_img = (noisy[i].squeeze() * 0.5 + 0.5).numpy()
        clean_img = (clean[i].squeeze() * 0.5 + 0.5).numpy()
        
        axes[0, i].imshow(noisy_img, cmap='gray')
        axes[0, i].set_title(f'Noisy\n{filenames[i][:20]}...')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(clean_img, cmap='gray')
        axes[1, i].set_title('Clean (Ground Truth)')
        axes[1, i].axis('off')
    
    plt.suptitle('Sample Noisy-Clean Image Pairs', fontsize=14)
    plt.tight_layout()
    plt.show()

# Visualize training samples
visualize_samples(train_loader)

## 5. Mamba SSM Building Blocks

### 5.1 Core Mamba Components
The Mamba architecture uses Selective State Space Models (S4/S6) with input-dependent selection mechanism for efficient sequence modeling.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class SelectiveSSM(nn.Module):
    """
    Selective State Space Model (S6) - Core Mamba mechanism.
    Implements input-dependent selection for efficient sequence modeling.
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dt_rank="auto"):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(expand * d_model)
        
        if dt_rank == "auto":
            self.dt_rank = max(d_model // 16, 1)
        else:
            self.dt_rank = dt_rank
        
        # Input projection
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        
        # Convolution for local context
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv, padding=d_conv - 1,
            groups=self.d_inner, bias=True
        )
        
        # SSM parameters projection
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        
        # Initialize dt projection bias
        dt_init_std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        
        # SSM state matrix A (diagonal)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        
        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
    
    def forward(self, x):
        """
        Args:
            x: (B, L, D) input tensor
        Returns:
            y: (B, L, D) output tensor
        """
        B, L, D = x.shape
        
        # Input projection
        xz = self.in_proj(x)  # (B, L, 2*d_inner)
        x, z = xz.chunk(2, dim=-1)  # Each (B, L, d_inner)
        
        # Convolution
        x = x.transpose(1, 2)  # (B, d_inner, L)
        x = self.conv1d(x)[:, :, :L]  # (B, d_inner, L)
        x = x.transpose(1, 2)  # (B, L, d_inner)
        x = F.silu(x)
        
        # SSM parameters from input
        x_proj = self.x_proj(x)  # (B, L, dt_rank + 2*d_state)
        dt, B_param, C = torch.split(
            x_proj, [self.dt_rank, self.d_state, self.d_state], dim=-1
        )
        
        # Compute dt
        dt = self.dt_proj(dt)  # (B, L, d_inner)
        dt = F.softplus(dt)
        
        # Get A from log
        A = -torch.exp(self.A_log)  # (d_inner, d_state)
        
        # Selective scan (simplified sequential version)
        y = self.selective_scan(x, dt, A, B_param, C)
        
        # Skip connection with D
        y = y + x * self.D
        
        # Gate with z
        y = y * F.silu(z)
        
        # Output projection
        y = self.out_proj(y)
        
        return y
    
    def selective_scan(self, x, dt, A, B, C):
        """
        Selective scan algorithm (sequential implementation).
        For production, use CUDA kernel from mamba-ssm package.
        """
        B_batch, L, d_inner = x.shape
        d_state = self.d_state
        
        # Discretize A and B
        # A: (d_inner, d_state), dt: (B, L, d_inner)
        dA = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))  # (B, L, d_inner, d_state)
        dB = dt.unsqueeze(-1) * B.unsqueeze(2)  # (B, L, d_inner, d_state)
        
        # Initialize state
        h = torch.zeros(B_batch, d_inner, d_state, device=x.device, dtype=x.dtype)
        
        # Sequential scan
        ys = []
        for i in range(L):
            h = dA[:, i] * h + dB[:, i] * x[:, i:i+1, :].transpose(1, 2)  # (B, d_inner, d_state)
            y = torch.einsum('bds,bds->bd', h, C[:, i:i+1, :].expand(-1, d_inner, -1).transpose(1, 2))
            ys.append(y)
        
        y = torch.stack(ys, dim=1)  # (B, L, d_inner)
        return y


class MambaBlock(nn.Module):
    """Single Mamba block with residual connection."""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.mamba = SelectiveSSM(d_model, d_state, d_conv, expand)
    
    def forward(self, x):
        return x + self.mamba(self.norm(x))


print("Mamba SSM building blocks defined successfully!")

### 5.2 Vision Mamba Components
Adapts Mamba for 2D image processing with patch embedding and bidirectional scanning.

In [ ]:
class PatchEmbed(nn.Module):
    """Convert image to patch embeddings."""
    def __init__(self, img_size=256, patch_size=4, in_chans=1, embed_dim=96):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x)  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)  # (B, n_patches, embed_dim)
        return x


class PatchExpand(nn.Module):
    """Expand patches back to image."""
    def __init__(self, dim, dim_out, scale=2):
        super().__init__()
        self.scale = scale
        self.expand = nn.Linear(dim, dim_out * scale * scale, bias=False)
        self.norm = nn.LayerNorm(dim_out)
    
    def forward(self, x, H, W):
        B, L, C = x.shape
        x = self.expand(x)
        x = x.view(B, H, W, -1)
        x = x.view(B, H, W, self.scale, self.scale, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
        x = x.view(B, H * self.scale, W * self.scale, -1)
        x = self.norm(x)
        x = x.view(B, -1, x.shape[-1])
        return x


class BiMambaBlock(nn.Module):
    """
    Bidirectional Mamba block for 2D images.
    Processes sequences in both forward and backward directions.
    """
    def __init__(self, dim, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.dim = dim
        
        # Forward and backward Mamba
        self.norm = RMSNorm(dim)
        self.mamba_forward = SelectiveSSM(dim, d_state, d_conv, expand)
        self.mamba_backward = SelectiveSSM(dim, d_state, d_conv, expand)
        
        # Fusion layer
        self.fusion = nn.Linear(dim * 2, dim)
    
    def forward(self, x):
        """
        Args:
            x: (B, L, D) patch sequence
        Returns:
            y: (B, L, D) processed sequence
        """
        residual = x
        x = self.norm(x)
        
        # Forward pass
        y_forward = self.mamba_forward(x)
        
        # Backward pass (reverse sequence)
        x_backward = torch.flip(x, dims=[1])
        y_backward = self.mamba_backward(x_backward)
        y_backward = torch.flip(y_backward, dims=[1])
        
        # Fuse forward and backward
        y = torch.cat([y_forward, y_backward], dim=-1)
        y = self.fusion(y)
        
        return residual + y


class VisionMambaLayer(nn.Module):
    """Vision Mamba layer with multiple BiMamba blocks."""
    def __init__(self, dim, depth=2, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.blocks = nn.ModuleList([
            BiMambaBlock(dim, d_state, d_conv, expand)
            for _ in range(depth)
        ])
    
    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x


print("Vision Mamba components defined successfully!")

## 6. MambaUNet Model Architecture
U-Net style architecture with Mamba blocks for image dehazing.

In [ ]:
class ConvBlock(nn.Module):
    """Convolutional block with normalization and activation."""
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size, padding=padding),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
    
    def forward(self, x):
        return self.conv(x)


class DownBlock(nn.Module):
    """Downsampling block with conv and Mamba."""
    def __init__(self, in_channels, out_channels, mamba_dim, d_state=16, depth=2):
        super().__init__()
        self.conv = ConvBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(2)
        
        # Mamba for spatial features
        self.to_mamba = nn.Linear(out_channels, mamba_dim)
        self.mamba = VisionMambaLayer(mamba_dim, depth=depth, d_state=d_state)
        self.from_mamba = nn.Linear(mamba_dim, out_channels)
    
    def forward(self, x):
        x = self.conv(x)
        
        # Apply Mamba on flattened spatial features
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)  # (B, H*W, C)
        x_mamba = self.to_mamba(x_flat)
        x_mamba = self.mamba(x_mamba)
        x_mamba = self.from_mamba(x_mamba)
        x_mamba = x_mamba.transpose(1, 2).view(B, C, H, W)
        
        x = x + x_mamba  # Residual connection
        
        return self.pool(x), x  # Return pooled and skip


class UpBlock(nn.Module):
    """Upsampling block with conv and Mamba."""
    def __init__(self, in_channels, out_channels, mamba_dim, d_state=16, depth=2):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_channels, out_channels)
        
        # Mamba for spatial features
        self.to_mamba = nn.Linear(out_channels, mamba_dim)
        self.mamba = VisionMambaLayer(mamba_dim, depth=depth, d_state=d_state)
        self.from_mamba = nn.Linear(mamba_dim, out_channels)
    
    def forward(self, x, skip):
        x = self.up(x)
        
        # Handle size mismatch
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        
        # Apply Mamba
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)
        x_mamba = self.to_mamba(x_flat)
        x_mamba = self.mamba(x_mamba)
        x_mamba = self.from_mamba(x_mamba)
        x_mamba = x_mamba.transpose(1, 2).view(B, C, H, W)
        
        x = x + x_mamba
        
        return x


class MambaUNet(nn.Module):
    """
    U-Net architecture with Mamba SSM blocks for echocardiography dehazing.
    
    Architecture:
    - Encoder: Conv blocks with Mamba for long-range dependencies
    - Bottleneck: Deep Mamba processing
    - Decoder: Upsampling with skip connections and Mamba
    """
    def __init__(self, in_channels=1, out_channels=1, base_dim=64, 
                 mamba_dim=256, d_state=16, depths=[2, 2, 2, 2]):
        super().__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        # Initial convolution
        self.init_conv = nn.Conv2d(in_channels, base_dim, kernel_size=7, padding=3)
        
        # Encoder
        self.down1 = DownBlock(base_dim, base_dim, mamba_dim, d_state, depths[0])
        self.down2 = DownBlock(base_dim, base_dim * 2, mamba_dim, d_state, depths[1])
        self.down3 = DownBlock(base_dim * 2, base_dim * 4, mamba_dim, d_state, depths[2])
        self.down4 = DownBlock(base_dim * 4, base_dim * 8, mamba_dim, d_state, depths[3])
        
        # Bottleneck with deeper Mamba processing
        self.bottleneck_conv = ConvBlock(base_dim * 8, base_dim * 16)
        self.bottleneck_to_mamba = nn.Linear(base_dim * 16, mamba_dim)
        self.bottleneck_mamba = VisionMambaLayer(mamba_dim, depth=4, d_state=d_state)
        self.bottleneck_from_mamba = nn.Linear(mamba_dim, base_dim * 16)
        
        # Decoder
        self.up1 = UpBlock(base_dim * 16, base_dim * 8, mamba_dim, d_state, depths[3])
        self.up2 = UpBlock(base_dim * 8, base_dim * 4, mamba_dim, d_state, depths[2])
        self.up3 = UpBlock(base_dim * 4, base_dim * 2, mamba_dim, d_state, depths[1])
        self.up4 = UpBlock(base_dim * 2, base_dim, mamba_dim, d_state, depths[0])
        
        # Output
        self.final_conv = nn.Sequential(
            nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_dim),
            nn.GELU(),
            nn.Conv2d(base_dim, out_channels, kernel_size=1),
            nn.Tanh()  # Output in [-1, 1] range
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        Args:
            x: (B, 1, H, W) noisy echocardiography image
        Returns:
            out: (B, 1, H, W) dehazed image
        """
        # Initial convolution
        x = self.init_conv(x)
        
        # Encoder
        x, skip1 = self.down1(x)
        x, skip2 = self.down2(x)
        x, skip3 = self.down3(x)
        x, skip4 = self.down4(x)
        
        # Bottleneck
        x = self.bottleneck_conv(x)
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)
        x_mamba = self.bottleneck_to_mamba(x_flat)
        x_mamba = self.bottleneck_mamba(x_mamba)
        x_mamba = self.bottleneck_from_mamba(x_mamba)
        x_mamba = x_mamba.transpose(1, 2).view(B, C, H, W)
        x = x + x_mamba
        
        # Decoder
        x = self.up1(x, skip4)
        x = self.up2(x, skip3)
        x = self.up3(x, skip2)
        x = self.up4(x, skip1)
        
        # Output
        out = self.final_conv(x)
        
        return out


# Create model
model = MambaUNet(
    in_channels=config.IN_CHANNELS,
    out_channels=config.IN_CHANNELS,
    base_dim=config.BASE_DIM,
    mamba_dim=config.MAMBA_DIM,
    d_state=config.SSM_STATE_DIM,
    depths=[2, 2, 2, 2]
).to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / 1e6:.2f} MB (float32)")

## 7. Loss Functions and Metrics

In [ ]:
class SSIMLoss(nn.Module):
    """Structural Similarity Index Loss."""
    def __init__(self, window_size=11, size_average=True):
        super().__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = 1
        self.window = self._create_window(window_size, self.channel)
    
    def _create_window(self, window_size, channel):
        def gaussian(window_size, sigma):
            gauss = torch.Tensor([
                np.exp(-(x - window_size // 2) ** 2 / (2 * sigma ** 2))
                for x in range(window_size)
            ])
            return gauss / gauss.sum()
        
        _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
        _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
        window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
        return window
    
    def _ssim(self, img1, img2, window, window_size, channel, size_average=True):
        mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
        mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)
        
        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2
        
        sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2
        
        C1 = 0.01 ** 2
        C2 = 0.03 ** 2
        
        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
                   ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
        
        if size_average:
            return ssim_map.mean()
        else:
            return ssim_map.mean(1).mean(1).mean(1)
    
    def forward(self, img1, img2):
        channel = img1.size(1)
        
        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = self._create_window(self.window_size, channel)
            window = window.to(img1.device).type(img1.dtype)
            self.window = window
            self.channel = channel
        
        return 1 - self._ssim(img1, img2, window, self.window_size, channel, self.size_average)


class PerceptualLoss(nn.Module):
    """Simple perceptual loss using gradient-based features."""
    def __init__(self):
        super().__init__()
        # Sobel filters for edge detection
        self.sobel_x = nn.Conv2d(1, 1, 3, padding=1, bias=False)
        self.sobel_y = nn.Conv2d(1, 1, 3, padding=1, bias=False)
        
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        
        self.sobel_x.weight = nn.Parameter(sobel_x.view(1, 1, 3, 3), requires_grad=False)
        self.sobel_y.weight = nn.Parameter(sobel_y.view(1, 1, 3, 3), requires_grad=False)
    
    def forward(self, pred, target):
        # Move filters to correct device
        self.sobel_x = self.sobel_x.to(pred.device)
        self.sobel_y = self.sobel_y.to(pred.device)
        
        # Compute gradients
        pred_grad_x = self.sobel_x(pred)
        pred_grad_y = self.sobel_y(pred)
        target_grad_x = self.sobel_x(target)
        target_grad_y = self.sobel_y(target)
        
        # L1 loss on gradients
        loss_x = F.l1_loss(pred_grad_x, target_grad_x)
        loss_y = F.l1_loss(pred_grad_y, target_grad_y)
        
        return loss_x + loss_y


class CombinedLoss(nn.Module):
    """Combined loss for dehazing: L1 + SSIM + Perceptual."""
    def __init__(self, l1_weight=1.0, ssim_weight=0.5, perceptual_weight=0.1):
        super().__init__()
        self.l1_weight = l1_weight
        self.ssim_weight = ssim_weight
        self.perceptual_weight = perceptual_weight
        
        self.l1_loss = nn.L1Loss()
        self.ssim_loss = SSIMLoss()
        self.perceptual_loss = PerceptualLoss()
    
    def forward(self, pred, target):
        l1 = self.l1_loss(pred, target)
        ssim = self.ssim_loss(pred, target)
        perceptual = self.perceptual_loss(pred, target)
        
        total = (self.l1_weight * l1 + 
                 self.ssim_weight * ssim + 
                 self.perceptual_weight * perceptual)
        
        return total, {
            'l1': l1.item(),
            'ssim': ssim.item(),
            'perceptual': perceptual.item(),
            'total': total.item()
        }


# Initialize loss function
criterion = CombinedLoss(
    l1_weight=config.L1_WEIGHT,
    ssim_weight=config.SSIM_WEIGHT,
    perceptual_weight=config.PERCEPTUAL_WEIGHT
)

print("Loss functions initialized!")

## 8. Evaluation Metrics

Implementing metrics from the DehazingEcho2025 challenge:
- PSNR (Peak Signal-to-Noise Ratio)
- SSIM (Structural Similarity Index)
- CNR (Contrast-to-Noise Ratio)
- FID (Fréchet Inception Distance) - simplified version

In [ ]:
class MetricsCalculator:
    """Calculate evaluation metrics for dehazing quality assessment."""
    
    @staticmethod
    def denormalize(tensor):
        """Convert from [-1, 1] to [0, 1] range."""
        return (tensor * 0.5 + 0.5).clamp(0, 1)
    
    @staticmethod
    def to_numpy(tensor):
        """Convert tensor to numpy array."""
        if isinstance(tensor, torch.Tensor):
            return tensor.detach().cpu().numpy()
        return tensor
    
    @staticmethod
    def calculate_psnr(pred, target):
        """Calculate PSNR between prediction and target."""
        pred = MetricsCalculator.denormalize(pred)
        target = MetricsCalculator.denormalize(target)
        
        pred_np = MetricsCalculator.to_numpy(pred.squeeze())
        target_np = MetricsCalculator.to_numpy(target.squeeze())
        
        return psnr(target_np, pred_np, data_range=1.0)
    
    @staticmethod
    def calculate_ssim(pred, target):
        """Calculate SSIM between prediction and target."""
        pred = MetricsCalculator.denormalize(pred)
        target = MetricsCalculator.denormalize(target)
        
        pred_np = MetricsCalculator.to_numpy(pred.squeeze())
        target_np = MetricsCalculator.to_numpy(target.squeeze())
        
        return ssim(target_np, pred_np, data_range=1.0)
    
    @staticmethod
    def calculate_cnr(image, roi1_mask=None, roi2_mask=None):
        """
        Calculate Contrast-to-Noise Ratio.
        If masks not provided, uses simple region-based estimation.
        """
        img = MetricsCalculator.denormalize(image)
        img_np = MetricsCalculator.to_numpy(img.squeeze())
        
        # Simple CNR estimation using image statistics
        # Divide image into regions
        h, w = img_np.shape
        center_region = img_np[h//4:3*h//4, w//4:3*w//4]
        border_region = np.concatenate([
            img_np[:h//4, :].flatten(),
            img_np[3*h//4:, :].flatten(),
            img_np[h//4:3*h//4, :w//4].flatten(),
            img_np[h//4:3*h//4, 3*w//4:].flatten()
        ])
        
        mean_center = np.mean(center_region)
        mean_border = np.mean(border_region)
        std_border = np.std(border_region) + 1e-8
        
        cnr = np.abs(mean_center - mean_border) / std_border
        return cnr
    
    @staticmethod
    def calculate_ks_statistic(pred, target):
        """
        Calculate Kolmogorov-Smirnov statistic for distribution similarity.
        Lower values indicate more similar distributions.
        """
        pred = MetricsCalculator.denormalize(pred)
        target = MetricsCalculator.denormalize(target)
        
        pred_np = MetricsCalculator.to_numpy(pred.flatten())
        target_np = MetricsCalculator.to_numpy(target.flatten())
        
        ks_stat, _ = stats.ks_2samp(pred_np, target_np)
        return ks_stat
    
    @staticmethod
    def calculate_mse(pred, target):
        """Calculate Mean Squared Error."""
        pred = MetricsCalculator.denormalize(pred)
        target = MetricsCalculator.denormalize(target)
        return F.mse_loss(pred, target).item()
    
    @staticmethod
    def calculate_mae(pred, target):
        """Calculate Mean Absolute Error."""
        pred = MetricsCalculator.denormalize(pred)
        target = MetricsCalculator.denormalize(target)
        return F.l1_loss(pred, target).item()
    
    @staticmethod
    def calculate_all_metrics(pred, target):
        """Calculate all metrics for a prediction-target pair."""
        metrics = {
            'psnr': MetricsCalculator.calculate_psnr(pred, target),
            'ssim': MetricsCalculator.calculate_ssim(pred, target),
            'cnr_pred': MetricsCalculator.calculate_cnr(pred),
            'cnr_target': MetricsCalculator.calculate_cnr(target),
            'ks_statistic': MetricsCalculator.calculate_ks_statistic(pred, target),
            'mse': MetricsCalculator.calculate_mse(pred, target),
            'mae': MetricsCalculator.calculate_mae(pred, target)
        }
        return metrics


def evaluate_model(model, dataloader, device, num_samples=None):
    """Evaluate model on a dataset and return metrics."""
    model.eval()
    all_metrics = []
    
    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc="Evaluating")):
            if num_samples and i >= num_samples:
                break
            
            noisy = batch['noisy'].to(device)
            clean = batch['clean'].to(device)
            
            # Generate dehazed output
            pred = model(noisy)
            
            # Calculate metrics for each image in batch
            for j in range(noisy.shape[0]):
                metrics = MetricsCalculator.calculate_all_metrics(
                    pred[j:j+1], clean[j:j+1]
                )
                metrics['filename'] = batch['filename'][j]
                all_metrics.append(metrics)
    
    # Aggregate metrics
    df = pd.DataFrame(all_metrics)
    summary = {
        'psnr_mean': df['psnr'].mean(),
        'psnr_std': df['psnr'].std(),
        'ssim_mean': df['ssim'].mean(),
        'ssim_std': df['ssim'].std(),
        'cnr_pred_mean': df['cnr_pred'].mean(),
        'ks_statistic_mean': df['ks_statistic'].mean(),
        'mse_mean': df['mse'].mean(),
        'mae_mean': df['mae'].mean()
    }
    
    return df, summary


print("Metrics calculator initialized!")

## 9. Training Loop

In [ ]:
class Trainer:
    """Training manager for MambaUNet dehazing model."""
    
    def __init__(self, model, criterion, optimizer, scheduler, device, config):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.config = config
        
        self.best_val_loss = float('inf')
        self.best_psnr = 0
        self.patience_counter = 0
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_psnr': [], 'val_psnr': [],
            'train_ssim': [], 'val_ssim': [],
            'lr': []
        }
    
    def train_epoch(self, dataloader):
        """Train for one epoch."""
        self.model.train()
        epoch_losses = []
        epoch_psnr = []
        epoch_ssim = []
        
        pbar = tqdm(dataloader, desc="Training")
        for batch_idx, batch in enumerate(pbar):
            noisy = batch['noisy'].to(self.device)
            clean = batch['clean'].to(self.device)
            
            # Forward pass
            self.optimizer.zero_grad()
            pred = self.model(noisy)
            loss, loss_dict = self.criterion(pred, clean)
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            # Calculate metrics
            with torch.no_grad():
                batch_psnr = np.mean([
                    MetricsCalculator.calculate_psnr(pred[i], clean[i])
                    for i in range(pred.shape[0])
                ])
                batch_ssim = np.mean([
                    MetricsCalculator.calculate_ssim(pred[i], clean[i])
                    for i in range(pred.shape[0])
                ])
            
            epoch_losses.append(loss.item())
            epoch_psnr.append(batch_psnr)
            epoch_ssim.append(batch_ssim)
            
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'psnr': f"{batch_psnr:.2f}",
                'ssim': f"{batch_ssim:.4f}"
            })
        
        return {
            'loss': np.mean(epoch_losses),
            'psnr': np.mean(epoch_psnr),
            'ssim': np.mean(epoch_ssim)
        }
    
    @torch.no_grad()
    def validate(self, dataloader):
        """Validate the model."""
        self.model.eval()
        epoch_losses = []
        epoch_psnr = []
        epoch_ssim = []
        
        for batch in tqdm(dataloader, desc="Validating"):
            noisy = batch['noisy'].to(self.device)
            clean = batch['clean'].to(self.device)
            
            pred = self.model(noisy)
            loss, _ = self.criterion(pred, clean)
            
            batch_psnr = np.mean([
                MetricsCalculator.calculate_psnr(pred[i], clean[i])
                for i in range(pred.shape[0])
            ])
            batch_ssim = np.mean([
                MetricsCalculator.calculate_ssim(pred[i], clean[i])
                for i in range(pred.shape[0])
            ])
            
            epoch_losses.append(loss.item())
            epoch_psnr.append(batch_psnr)
            epoch_ssim.append(batch_ssim)
        
        return {
            'loss': np.mean(epoch_losses),
            'psnr': np.mean(epoch_psnr),
            'ssim': np.mean(epoch_ssim)
        }
    
    def save_checkpoint(self, epoch, is_best=False):
        """Save model checkpoint."""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_val_loss': self.best_val_loss,
            'best_psnr': self.best_psnr,
            'history': self.history,
            'config': vars(self.config)
        }
        
        # Save latest checkpoint
        torch.save(checkpoint, os.path.join(self.config.CHECKPOINT_DIR, 'latest.pt'))
        
        # Save best checkpoint
        if is_best:
            torch.save(checkpoint, os.path.join(self.config.CHECKPOINT_DIR, 'best.pt'))
            print(f"  ✓ Saved best model (PSNR: {self.best_psnr:.2f})")
    
    def load_checkpoint(self, checkpoint_path):
        """Load model checkpoint."""
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.best_val_loss = checkpoint['best_val_loss']
        self.best_psnr = checkpoint['best_psnr']
        self.history = checkpoint['history']
        return checkpoint['epoch']
    
    def train(self, train_loader, val_loader, start_epoch=0):
        """Full training loop."""
        print(f"\n{'='*60}")
        print(f"Starting Training")
        print(f"{'='*60}")
        print(f"Epochs: {self.config.NUM_EPOCHS}")
        print(f"Batch size: {self.config.BATCH_SIZE}")
        print(f"Learning rate: {self.config.LEARNING_RATE}")
        print(f"Device: {self.device}")
        print(f"{'='*60}\n")
        
        for epoch in range(start_epoch, self.config.NUM_EPOCHS):
            print(f"\nEpoch {epoch + 1}/{self.config.NUM_EPOCHS}")
            print("-" * 40)
            
            # Training
            train_metrics = self.train_epoch(train_loader)
            
            # Validation
            val_metrics = self.validate(val_loader)
            
            # Update learning rate
            current_lr = self.optimizer.param_groups[0]['lr']
            self.scheduler.step(val_metrics['loss'])
            
            # Log metrics
            self.history['train_loss'].append(train_metrics['loss'])
            self.history['val_loss'].append(val_metrics['loss'])
            self.history['train_psnr'].append(train_metrics['psnr'])
            self.history['val_psnr'].append(val_metrics['psnr'])
            self.history['train_ssim'].append(train_metrics['ssim'])
            self.history['val_ssim'].append(val_metrics['ssim'])
            self.history['lr'].append(current_lr)
            
            # Print epoch summary
            print(f"\n  Train - Loss: {train_metrics['loss']:.4f}, "
                  f"PSNR: {train_metrics['psnr']:.2f}, SSIM: {train_metrics['ssim']:.4f}")
            print(f"  Val   - Loss: {val_metrics['loss']:.4f}, "
                  f"PSNR: {val_metrics['psnr']:.2f}, SSIM: {val_metrics['ssim']:.4f}")
            print(f"  LR: {current_lr:.2e}")
            
            # Check for improvement
            is_best = val_metrics['psnr'] > self.best_psnr
            if is_best:
                self.best_psnr = val_metrics['psnr']
                self.best_val_loss = val_metrics['loss']
                self.patience_counter = 0
            else:
                self.patience_counter += 1
            
            # Save checkpoint
            self.save_checkpoint(epoch, is_best)
            
            # Early stopping
            if self.patience_counter >= self.config.EARLY_STOP_PATIENCE:
                print(f"\nEarly stopping triggered after {epoch + 1} epochs")
                break
        
        print(f"\n{'='*60}")
        print(f"Training Complete!")
        print(f"Best Validation PSNR: {self.best_psnr:.2f}")
        print(f"{'='*60}")
        
        return self.history


# Setup training
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=config.SCHEDULER_PATIENCE,
    verbose=True
)

trainer = Trainer(model, criterion, optimizer, scheduler, device, config)

print("Trainer initialized and ready!")

## 10. Start Training

⚠️ **Note**: Training may take several hours depending on your hardware. The model will save checkpoints automatically.

In [ ]:
# Check for existing checkpoint to resume training
checkpoint_path = os.path.join(config.CHECKPOINT_DIR, 'latest.pt')
start_epoch = 0

if os.path.exists(checkpoint_path):
    print(f"Found existing checkpoint: {checkpoint_path}")
    response = input("Resume training? (y/n): ").lower()
    if response == 'y':
        start_epoch = trainer.load_checkpoint(checkpoint_path) + 1
        print(f"Resuming from epoch {start_epoch}")

# Start training
history = trainer.train(train_loader, val_loader, start_epoch=start_epoch)

## 11. Training Visualization

In [ ]:
def plot_training_history(history, save_path=None):
    """Plot training history curves."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train')
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Validation')
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # PSNR
    axes[0, 1].plot(epochs, history['train_psnr'], 'b-', label='Train')
    axes[0, 1].plot(epochs, history['val_psnr'], 'r-', label='Validation')
    axes[0, 1].set_title('PSNR (dB)')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('PSNR')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # SSIM
    axes[1, 0].plot(epochs, history['train_ssim'], 'b-', label='Train')
    axes[1, 0].plot(epochs, history['val_ssim'], 'r-', label='Validation')
    axes[1, 0].set_title('SSIM')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('SSIM')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Learning Rate
    axes[1, 1].plot(epochs, history['lr'], 'g-')
    axes[1, 1].set_title('Learning Rate')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('LR')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True)
    
    plt.suptitle('Training History - Mamba SSM Dehazing', fontsize=14)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Training history plot saved to: {save_path}")
    
    plt.show()

# Plot training history
if history:
    plot_training_history(
        history, 
        save_path=os.path.join(config.RESULTS_DIR, 'training_history.png')
    )

## 12. Load Best Model and Evaluate on Test Set

In [ ]:
# Load best model
best_checkpoint_path = os.path.join(config.CHECKPOINT_DIR, 'best.pt')

if os.path.exists(best_checkpoint_path):
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")
    print(f"Best validation PSNR: {checkpoint['best_psnr']:.2f}")
else:
    print("No best checkpoint found. Using current model.")

# Evaluate on test set
print("\n" + "="*60)
print("Evaluating on Test Set")
print("="*60)

test_df, test_summary = evaluate_model(model, test_loader, device)

# Print summary
print("\n📊 Test Set Evaluation Results:")
print("-" * 40)
print(f"PSNR:  {test_summary['psnr_mean']:.2f} ± {test_summary['psnr_std']:.2f} dB")
print(f"SSIM:  {test_summary['ssim_mean']:.4f} ± {test_summary['ssim_std']:.4f}")
print(f"CNR:   {test_summary['cnr_pred_mean']:.4f}")
print(f"KS:    {test_summary['ks_statistic_mean']:.4f}")
print(f"MSE:   {test_summary['mse_mean']:.6f}")
print(f"MAE:   {test_summary['mae_mean']:.6f}")

# Save detailed results
results_csv_path = os.path.join(config.RESULTS_DIR, 'test_results.csv')
test_df.to_csv(results_csv_path, index=False)
print(f"\nDetailed results saved to: {results_csv_path}")

## 13. Visualize Dehazing Results

In [ ]:
def visualize_dehazing_results(model, dataloader, device, num_samples=6, save_path=None):
    """
    Visualize dehazing results in a 3-column grid:
    Noisy (Input) | Output (Dehazed) | Ground Truth
    With labels on top and metrics on bottom.
    """
    model.eval()
    
    batch = next(iter(dataloader))
    noisy = batch['noisy'][:num_samples].to(device)
    clean = batch['clean'][:num_samples].to(device)
    filenames = batch['filename'][:num_samples]
    
    with torch.no_grad():
        dehazed = model(noisy)
    
    # Create figure: num_samples rows x 3 columns
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    
    # Add column headers
    col_titles = ['Noisy (Input)', 'Output (Dehazed)', 'Ground Truth']
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=14, fontweight='bold', pad=10)
    
    for i in range(num_samples):
        # Denormalize for visualization
        noisy_img = (noisy[i].cpu().squeeze() * 0.5 + 0.5).numpy()
        dehazed_img = (dehazed[i].cpu().squeeze() * 0.5 + 0.5).clamp(0, 1).numpy()
        clean_img = (clean[i].cpu().squeeze() * 0.5 + 0.5).numpy()
        
        # Calculate metrics for this image
        psnr_val = MetricsCalculator.calculate_psnr(dehazed[i:i+1], clean[i:i+1])
        ssim_val = MetricsCalculator.calculate_ssim(dehazed[i:i+1], clean[i:i+1])
        mse_val = MetricsCalculator.calculate_mse(dehazed[i:i+1], clean[i:i+1])
        
        # Column 1: Noisy Input
        axes[i, 0].imshow(noisy_img, cmap='gray', vmin=0, vmax=1)
        axes[i, 0].axis('off')
        axes[i, 0].set_xlabel(f'Sample {i+1}: {filenames[i][:25]}...', fontsize=9)
        
        # Column 2: Dehazed Output
        axes[i, 1].imshow(dehazed_img, cmap='gray', vmin=0, vmax=1)
        axes[i, 1].axis('off')
        
        # Column 3: Ground Truth
        axes[i, 2].imshow(clean_img, cmap='gray', vmin=0, vmax=1)
        axes[i, 2].axis('off')
        
        # Add metrics below the row (centered under the 3 images)
        metrics_text = f'PSNR: {psnr_val:.2f} dB  |  SSIM: {ssim_val:.4f}  |  MSE: {mse_val:.6f}'
        fig.text(
            0.5, 1 - (i + 1) / num_samples + 0.01,
            metrics_text,
            ha='center', va='top',
            fontsize=10, color='darkblue',
            transform=fig.transFigure
        )
    
    plt.suptitle('Mamba SSM Echocardiography Dehazing Results', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.15, wspace=0.05)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"✓ Results visualization saved to: {save_path}")
    
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*60)
    print("SUMMARY METRICS FOR VISUALIZED SAMPLES")
    print("="*60)
    
    all_psnr, all_ssim = [], []
    for i in range(num_samples):
        psnr_val = MetricsCalculator.calculate_psnr(dehazed[i:i+1], clean[i:i+1])
        ssim_val = MetricsCalculator.calculate_ssim(dehazed[i:i+1], clean[i:i+1])
        all_psnr.append(psnr_val)
        all_ssim.append(ssim_val)
        print(f"  Sample {i+1}: PSNR = {psnr_val:.2f} dB, SSIM = {ssim_val:.4f}")
    
    print("-"*60)
    print(f"  Average:  PSNR = {np.mean(all_psnr):.2f} ± {np.std(all_psnr):.2f} dB")
    print(f"            SSIM = {np.mean(all_ssim):.4f} ± {np.std(all_ssim):.4f}")
    print("="*60)


# Visualize test results
visualize_dehazing_results(
    model, test_loader, device, num_samples=6,
    save_path=os.path.join(config.RESULTS_DIR, 'dehazing_results.png')
)

## 14. Generate Dehazed Images for All Test Samples

In [ ]:
def generate_dehazed_images(model, dataloader, device, output_dir):
    """Generate dehazed images for all samples in dataloader."""
    model.eval()
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Generating dehazed images to: {output_dir}")
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Generating"):
            noisy = batch['noisy'].to(device)
            filenames = batch['filename']
            
            # Generate dehazed images
            dehazed = model(noisy)
            
            # Save each image
            for i, filename in enumerate(filenames):
                # Denormalize to [0, 1]
                img = (dehazed[i].cpu().squeeze() * 0.5 + 0.5).clamp(0, 1)
                
                # Convert to PIL and save
                img_pil = transforms.ToPILImage()(img)
                save_path = os.path.join(output_dir, filename)
                img_pil.save(save_path)
    
    print(f"✓ Generated {len(dataloader.dataset)} dehazed images")


# Create output directory for dehazed images
dehazed_output_dir = os.path.join(config.RESULTS_DIR, 'dehazed_images')
generate_dehazed_images(model, test_loader, device, dehazed_output_dir)

## 15. Metrics Distribution Analysis

In [ ]:
def plot_metrics_distribution(test_df, save_path=None):
    """Plot distribution of evaluation metrics."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # PSNR distribution
    axes[0, 0].hist(test_df['psnr'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(test_df['psnr'].mean(), color='red', linestyle='--', label=f'Mean: {test_df["psnr"].mean():.2f}')
    axes[0, 0].set_title('PSNR Distribution')
    axes[0, 0].set_xlabel('PSNR (dB)')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].legend()
    
    # SSIM distribution
    axes[0, 1].hist(test_df['ssim'], bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
    axes[0, 1].axvline(test_df['ssim'].mean(), color='red', linestyle='--', label=f'Mean: {test_df["ssim"].mean():.4f}')
    axes[0, 1].set_title('SSIM Distribution')
    axes[0, 1].set_xlabel('SSIM')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].legend()
    
    # CNR distribution
    axes[0, 2].hist(test_df['cnr_pred'], bins=30, color='darkorange', edgecolor='black', alpha=0.7)
    axes[0, 2].axvline(test_df['cnr_pred'].mean(), color='red', linestyle='--', label=f'Mean: {test_df["cnr_pred"].mean():.4f}')
    axes[0, 2].set_title('CNR Distribution (Predicted)')
    axes[0, 2].set_xlabel('CNR')
    axes[0, 2].set_ylabel('Count')
    axes[0, 2].legend()
    
    # KS Statistic distribution
    axes[1, 0].hist(test_df['ks_statistic'], bins=30, color='purple', edgecolor='black', alpha=0.7)
    axes[1, 0].axvline(test_df['ks_statistic'].mean(), color='red', linestyle='--', label=f'Mean: {test_df["ks_statistic"].mean():.4f}')
    axes[1, 0].set_title('KS Statistic Distribution')
    axes[1, 0].set_xlabel('KS Statistic')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].legend()
    
    # MSE distribution
    axes[1, 1].hist(test_df['mse'], bins=30, color='crimson', edgecolor='black', alpha=0.7)
    axes[1, 1].axvline(test_df['mse'].mean(), color='blue', linestyle='--', label=f'Mean: {test_df["mse"].mean():.6f}')
    axes[1, 1].set_title('MSE Distribution')
    axes[1, 1].set_xlabel('MSE')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].legend()
    
    # PSNR vs SSIM scatter
    axes[1, 2].scatter(test_df['psnr'], test_df['ssim'], alpha=0.5, c='teal')
    axes[1, 2].set_title('PSNR vs SSIM')
    axes[1, 2].set_xlabel('PSNR (dB)')
    axes[1, 2].set_ylabel('SSIM')
    
    plt.suptitle('Evaluation Metrics Distribution - Mamba SSM Dehazing', fontsize=14)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Metrics distribution plot saved to: {save_path}")
    
    plt.show()


# Plot metrics distribution
if 'test_df' in dir() and len(test_df) > 0:
    plot_metrics_distribution(
        test_df,
        save_path=os.path.join(config.RESULTS_DIR, 'metrics_distribution.png')
    )

## 16. Export Model for Inference

In [ ]:
def export_model(model, config, save_dir):
    """Export model for inference with all necessary information."""
    os.makedirs(save_dir, exist_ok=True)
    
    # Save model weights only
    model_path = os.path.join(save_dir, 'mamba_dehazing_model.pt')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': vars(config),
        'model_class': 'MambaUNet',
        'input_size': config.IMG_SIZE,
        'in_channels': config.IN_CHANNELS,
    }, model_path)
    print(f"Model saved to: {model_path}")
    
    # Save model architecture summary
    summary_path = os.path.join(save_dir, 'model_summary.txt')
    with open(summary_path, 'w') as f:
        f.write("Mamba SSM Dehazing Model Summary\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Architecture: MambaUNet with Vision Mamba Blocks\n")
        f.write(f"Input Size: {config.IMG_SIZE} x {config.IMG_SIZE}\n")
        f.write(f"Input Channels: {config.IN_CHANNELS}\n")
        f.write(f"Base Dimension: {config.BASE_DIM}\n")
        f.write(f"Mamba Dimension: {config.MAMBA_DIM}\n")
        f.write(f"SSM State Dimension: {config.SSM_STATE_DIM}\n\n")
        
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        f.write(f"Total Parameters: {total_params:,}\n")
        f.write(f"Trainable Parameters: {trainable_params:,}\n")
        f.write(f"Model Size: {total_params * 4 / 1e6:.2f} MB\n\n")
        
        f.write("Model Architecture:\n")
        f.write(str(model))
    
    print(f"Model summary saved to: {summary_path}")
    
    # Create inference script
    inference_script = '''
"""
Mamba SSM Dehazing - Inference Script
=====================================
Use this script to dehaze echocardiography images.

Usage:
    python inference.py --input <input_image_or_dir> --output <output_dir>
"""

import os
import torch
import argparse
from PIL import Image
from torchvision import transforms
from glob import glob

# Import model architecture (copy MambaUNet class here or import)

def load_model(checkpoint_path, device):
    """Load trained model."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Recreate model with saved config
    config = checkpoint['config']
    model = MambaUNet(
        in_channels=config['IN_CHANNELS'],
        out_channels=config['IN_CHANNELS'],
        base_dim=config['BASE_DIM'],
        mamba_dim=config['MAMBA_DIM'],
        d_state=config['SSM_STATE_DIM']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model, config

def dehaze_image(model, image_path, output_path, img_size=256, device='cuda'):
    """Dehaze a single image."""
    # Load and preprocess
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    img = Image.open(image_path).convert('L')
    original_size = img.size
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    # Dehaze
    with torch.no_grad():
        dehazed = model(img_tensor)
    
    # Postprocess
    dehazed = (dehazed.squeeze().cpu() * 0.5 + 0.5).clamp(0, 1)
    dehazed_pil = transforms.ToPILImage()(dehazed)
    dehazed_pil = dehazed_pil.resize(original_size, Image.LANCZOS)
    dehazed_pil.save(output_path)

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Mamba SSM Dehazing Inference')
    parser.add_argument('--input', type=str, required=True, help='Input image or directory')
    parser.add_argument('--output', type=str, required=True, help='Output directory')
    parser.add_argument('--checkpoint', type=str, default='mamba_dehazing_model.pt')
    parser.add_argument('--device', type=str, default='cuda')
    args = parser.parse_args()
    
    device = torch.device(args.device if torch.cuda.is_available() else 'cpu')
    model, config = load_model(args.checkpoint, device)
    
    os.makedirs(args.output, exist_ok=True)
    
    if os.path.isfile(args.input):
        images = [args.input]
    else:
        images = glob(os.path.join(args.input, '*.png')) + glob(os.path.join(args.input, '*.jpg'))
    
    for img_path in images:
        filename = os.path.basename(img_path)
        output_path = os.path.join(args.output, filename)
        dehaze_image(model, img_path, output_path, config['IMG_SIZE'], device)
        print(f'Processed: {filename}')
'''
    
    script_path = os.path.join(save_dir, 'inference.py')
    with open(script_path, 'w') as f:
        f.write(inference_script)
    print(f"Inference script saved to: {script_path}")
    
    return model_path


# Export model
export_path = export_model(model, config, config.CHECKPOINT_DIR)

## 17. Final Summary Report

In [ ]:
def generate_summary_report(config, test_summary, history, save_path=None):
    """Generate a comprehensive summary report."""
    
    report = f"""
{'='*70}
MAMBA SSM ECHOCARDIOGRAPHY DEHAZING - FINAL REPORT
{'='*70}

📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

1. MODEL ARCHITECTURE
{'-'*40}
• Architecture: MambaUNet with Bidirectional Vision Mamba Blocks
• Input Size: {config.IMG_SIZE} x {config.IMG_SIZE}
• Input Channels: {config.IN_CHANNELS} (Grayscale)
• Base Dimension: {config.BASE_DIM}
• Mamba Dimension: {config.MAMBA_DIM}
• SSM State Dimension: {config.SSM_STATE_DIM}
• Total Parameters: {sum(p.numel() for p in model.parameters()):,}

2. TRAINING CONFIGURATION
{'-'*40}
• Batch Size: {config.BATCH_SIZE}
• Learning Rate: {config.LEARNING_RATE}
• Epochs Trained: {len(history['train_loss']) if history else 'N/A'}
• Loss Function: L1 + SSIM + Perceptual (Edge-based)
• Optimizer: AdamW with weight decay {config.WEIGHT_DECAY}
• Scheduler: ReduceLROnPlateau (patience={config.SCHEDULER_PATIENCE})

3. DATASET INFORMATION
{'-'*40}
• Input Directory: {config.NOISY_DIR}
• Ground Truth Directory: {config.CLEAN_DIR}
• Train/Val/Test Split: {config.TRAIN_RATIO}/{config.VAL_RATIO}/{config.TEST_RATIO}

4. TEST SET EVALUATION METRICS
{'-'*40}
• PSNR:         {test_summary['psnr_mean']:.2f} ± {test_summary['psnr_std']:.2f} dB
• SSIM:         {test_summary['ssim_mean']:.4f} ± {test_summary['ssim_std']:.4f}
• CNR:          {test_summary['cnr_pred_mean']:.4f}
• KS Statistic: {test_summary['ks_statistic_mean']:.4f}
• MSE:          {test_summary['mse_mean']:.6f}
• MAE:          {test_summary['mae_mean']:.6f}

5. TRAINING PROGRESS
{'-'*40}
• Best Validation PSNR: {max(history['val_psnr']) if history and history['val_psnr'] else 'N/A':.2f} dB
• Best Validation SSIM: {max(history['val_ssim']) if history and history['val_ssim'] else 'N/A':.4f}
• Final Training Loss: {history['train_loss'][-1] if history and history['train_loss'] else 'N/A':.4f}
• Final Validation Loss: {history['val_loss'][-1] if history and history['val_loss'] else 'N/A':.4f}

6. OUTPUT FILES
{'-'*40}
• Best Model Checkpoint: {os.path.join(config.CHECKPOINT_DIR, 'best.pt')}
• Latest Checkpoint: {os.path.join(config.CHECKPOINT_DIR, 'latest.pt')}
• Test Results CSV: {os.path.join(config.RESULTS_DIR, 'test_results.csv')}
• Dehazed Images: {os.path.join(config.RESULTS_DIR, 'dehazed_images')}
• Training History Plot: {os.path.join(config.RESULTS_DIR, 'training_history.png')}

7. KEY FEATURES OF MAMBA SSM APPROACH
{'-'*40}
• Selective State Space Models for efficient long-range dependencies
• Bidirectional scanning for comprehensive spatial feature extraction
• U-Net architecture with skip connections for detail preservation
• Combined loss function (L1 + SSIM + Perceptual) for visual quality

{'='*70}
"""
    
    print(report)
    
    if save_path:
        with open(save_path, 'w') as f:
            f.write(report)
        print(f"\n📄 Report saved to: {save_path}")
    
    return report


# Generate final report
report = generate_summary_report(
    config, 
    test_summary if 'test_summary' in dir() else {
        'psnr_mean': 0, 'psnr_std': 0, 'ssim_mean': 0, 'ssim_std': 0,
        'cnr_pred_mean': 0, 'ks_statistic_mean': 0, 'mse_mean': 0, 'mae_mean': 0
    },
    history if 'history' in dir() else None,
    save_path=os.path.join(config.RESULTS_DIR, 'final_report.txt')
)

---

## 🎉 Notebook Complete!

### Quick Start Guide:
1. **Run cells 1-5** to set up dependencies and load data
2. **Run cells 6-9** to define the Mamba SSM model architecture  
3. **Run cells 10-11** to set up loss functions and metrics
4. **Run cell 12** to initialize and start training
5. **Run cells 13-17** to evaluate and visualize results

### Key Outputs:
- **Model checkpoint**: `checkpoints/mamba_dehazing/best.pt`
- **Test results**: `results/mamba_dehazing/test_results.csv`
- **Dehazed images**: `results/mamba_dehazing/dehazed_images/`

### Tips for Best Results:
- Increase `NUM_EPOCHS` for better convergence
- Adjust `BASE_DIM` and `MAMBA_DIM` based on available GPU memory
- Use early stopping to prevent overfitting
- Monitor CNR and SSIM metrics for dehazing quality